In [ ]:
import datetime
import pandas as pd
import my_garmin_common

# ==========================================
# 👇 ここに計測した数値を入力してください
# ==========================================
input_data = {
    'weight':       60.5,   # 体重 (kg)
    'bmi':          22.0,   # BMI
    'body_fat_pct': 18.5,   # 体脂肪率 (%)
    'muscle_pct':   35.0,   # 骨格筋率 (%)
    'bone_mass':    2.5,    # 骨量 (kg)
    'metabolism':   1500,   # 基礎代謝 (kcal)
    'visceral_fat': 3.0,    # 内臓脂肪レベル
}
# ==========================================

# --- 設定項目 ---
SPREADSHEET_ID = '15CCDjcBCqSWYacPWf_RNXTBdJZ33x6pXAc1PhwPfkiY'
SA_KEY_VALUE = my_garmin_common.get_secret("SA_KEY")

def main():
    """メイン処理"""
    if not SA_KEY_VALUE:
        print("❌ シークレット 'SA_KEY' が設定されていません。")
        return

    client = my_garmin_common.get_google_creds(SA_KEY_VALUE)
    if not client:
        return

    try:
        # 1. スプレッドシート接続
        spreadsheet = client.open_by_key(SPREADSHEET_ID)
        log_sheet = spreadsheet.sheet1 # ログ用のシート(Sheet1)を取得
        
        # daily_summaryシート（マスタ）の取得
        SUMMARY_SHEET_NAME = 'daily_summary'
        try:
            summary_sheet = spreadsheet.worksheet(SUMMARY_SHEET_NAME)
        except:
            print(f"Creating new sheet: {SUMMARY_SHEET_NAME}")
            summary_sheet = spreadsheet.add_worksheet(title=SUMMARY_SHEET_NAME, rows=1000, cols=20)
    except Exception as e:
        print(f"❌ Spreadsheet Connection Error: {e}")
        return

    # 2. 書き込むデータを作成
    today_str = datetime.date.today().strftime("%Y-%m-%d")

    # garmin_sync.pyのdata_dictとキーの順番・名前を完全に一致させる
    data_dict = {
        'calendarDate': today_str,
        # --- 活動量 (None) ---
        'steps': None,
        'distance_m': None,
        'floors_ascended': None,
        'active_calories': None,
        'total_calories': None,
        # --- 心拍・ストレス (None) ---
        'heart_rate': None,
        'max_heart_rate': None,
        'min_heart_rate': None,
        'stress': None,
        'body_battery': None,
        # --- 睡眠・回復 (None) ---
        'sleep_hours': None,
        # --- 体組成 (ここに入力値をセット) ---
        'weight': input_data.get('weight'),
        'bmi': input_data.get('bmi'),
        'body_fat_pct': input_data.get('body_fat_pct'),
        'muscle_pct': input_data.get('muscle_pct'),
        'visceral_fat': input_data.get('visceral_fat'),
        'metabolism': input_data.get('metabolism'),
        'bone_mass': input_data.get('bone_mass'),
        # --- その他 (None) ---
        'water_ml': None,
        'moderate_minutes': None,
        'vigorous_minutes': None,
    }

    # -------------------------------------------------
    # 3. Daily Summary (マスタ) の更新
    # -------------------------------------------------
    print("Updating daily_summary...")
    try:
        # 既存データを取得
        df_current = my_garmin_common.sheet_to_df(summary_sheet)
        
        # 新しいデータをDataFrame化
        df_new = pd.DataFrame([data_dict])
        df_new['calendarDate'] = df_new['calendarDate'].astype(str)
        df_new = df_new.set_index('calendarDate')
        
        # マージ (入力した項目だけが上書きされ、Garminデータ(Noneの部分)は維持される)
        df_updated = df_new.combine_first(df_current)
        df_updated = df_updated.sort_index(ascending=False)
        
        # 保存
        my_garmin_common.df_to_sheet(summary_sheet, df_updated)
    except Exception as e:
        print(f"⚠️ Daily Summary Update Failed: {e}")

    # -------------------------------------------------
    # 4. Log Sheet (履歴) の追記
    # -------------------------------------------------
    print("Appending to Log Sheet...")
    # ヘッダー更新（garmin_sync.pyと合わせる）
    headers = list(data_dict.keys())
    headers[0] = 'timestamp'
    column_map = {
        'timestamp': '実行日時',
        'calendarDate': '対象日付',
        'steps': '歩数',
        'distance_m': '移動距離',
        'floors_ascended': '上昇階数',
        'active_calories': '活動カロリー',
        'total_calories': '総カロリー',
        'heart_rate': '安静時心拍',
        'max_heart_rate': '最大心拍',
        'min_heart_rate': '最小心拍',
        'stress': 'ストレス',
        'body_battery': 'BodyBattery',
        'sleep_hours': '睡眠時間',
        'weight': '体重',
        'bmi': 'BMI',
        'body_fat_pct': '体脂肪率',
        'muscle_pct': '筋肉率',
        'visceral_fat': '内臓脂肪',
        'metabolism': '基礎代謝',
        'bone_mass': '骨量',
        'water_ml': '水分摂取',
        'moderate_minutes': '中強度運動',
        'vigorous_minutes': '高強度運動',
    }
    header_row = [f"{column_map.get(k, k)}({k})" for k in headers]
    try:
        log_sheet.update('A1', [header_row])
    except Exception as e:
        print(f"⚠️ Header Update Failed: {e}")

    log_values = list(data_dict.values())
    
    # 先頭の 'calendarDate' を実行日時に置き換える (garmin_sync.pyのログと仕様を合わせる)
    log_timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_values[0] = log_timestamp

    try:
        log_sheet.append_row(log_values)
        print(f"✅ Log sheet updated: {today_str}")
        print(f"   📊 {input_data}")
    except Exception as e:
        print(f"❌ Log Sheet Append Error: {e}")

if __name__ == "__main__":
    main()